In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from groq import Groq
api_key=os.getenv("api_key")
GROQ_MODEL= "llama-3.3-70b-versatile"
client=Groq(api_key=api_key)

import json
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

# For Api Rate limiting
import logging
from tenacity import (
     retry,  # Decorator that wraps a function with retry logic
    stop_after_attempt,  # Stop after N total attempts
    wait_exponential,  # Wait 1s, 2s, 4s, 8s between retries
    before_sleep_log, 

)


logging.basicConfig(
    level=logging.INFO,
    filename=r"..\logs\sindhu_rag.log",
    filemode="w",

)

logger = logging.getLogger(__name__)
attempt_counter={"n":0}


# Declaring some important variable 
Embendding_Model_Name='thenlper/gte-large'
Chroma_Path=r"..\Storage\sindhu_db"
        
Top_k=3# retrieving only 3 chunks because of api costs
collection="sindh-10k-collection"
embeddings=SentenceTransformerEmbeddings(model_name=Embendding_Model_Name)



C:\Users\Lenovo\AppData\Local\Temp\ipykernel_10220\333031106.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_10220\333031106.py:41: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=SentenceTransformerEmbeddings(model_name=Embendding_Model_Name)
d:\Sindhu-RAG\sindhu_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See 

In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent


CONVERSATION_PATH = PROJECT_ROOT / "Storage" / "conversation" / "conversation_history.json"

In [4]:
vectore_Store=Chroma(
    collection_name=collection,
    persist_directory=Chroma_Path,
    embedding_function=embeddings
)
# declaring the retreiver
retriever=vectore_Store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":Top_k}
)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_10220\1769221126.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectore_Store=Chroma(


In [5]:
# function for retreiving the chunks and returing it in the format of list of dictionaries
def retrieve_chunks(user_query,retriever):
    docs=retriever.invoke(user_query)
    retreived=[]
    for i,doc in enumerate(docs):
        retreived.append({
            "Index":i,
            "Text":doc.page_content,
            "Metadata":doc.metadata
        })
    return retreived



In [6]:
def loading_bundle(path):
    with open(path,"r") as f:
        dict=json.load(f)
    return dict

path=r"..\config\v2_config.json"
bundle=loading_bundle(path)

def loading_prompt(bundle):
    system_path=bundle["prompt"]
    with open(system_path,"r") as f:
        p=f.read()
    return p



In [ ]:
# --- Configuration ---
HISTORY_FILE = CONVERSATION_PATH
#HISTORY_FILE = r"..\Storage\conversation\conversation_history.json"
MAX_MESSAGE=2


# --- Helper functions for managing conversation history ---
def load_conversation_history():
    """Loads conversation history from a JSON file, or initializes it with the system message."""

    with open(HISTORY_FILE, 'r') as f:
        history = json.load(f)
      
        print(f"Loaded conversation history from {HISTORY_FILE}. Current turns: {len(history)-1}")
    if len(history)<MAX_MESSAGE:
        return history
    dropped=history[:-MAX_MESSAGE] # from top to -3 is dropped
    kept=history[-MAX_MESSAGE:] # from -3 to end is kept
    return kept
display(load_conversation_history())

Loaded conversation history from d:\Sindhu-RAG\Storage\conversation\conversation_history.json. Current turns: 6


[{'role': 'assistant',
  'content': 'The streets of Mohanjodaro were laid out in a grid pattern with main streets running north-south and east-west, dividing the area into blocks of roughly equal size and approximately rectangular shape, with the main streets being about 30 feet wide. The major insulae or blocks were subdivided by lanes which were not infrequently dog-legged, and these lanes were normally from 5 to 10 feet wide.'}]

In [8]:
# Building the system Prompt
SYSTEM_MESSAGE = loading_prompt(bundle).strip()
# building the context text from the retreived chunks
def build_context_block(retrieve_chunks):
    parts=[]
    for chunk in retrieve_chunks:
        parts.append(chunk["Text"])

    return "\n\n".join(parts)
# building the user query from the by atatching the context 
def build_user_message(user_query,retrieve_chunks):
    context_text=build_context_block(retrieve_chunks)
    conversation_history=load_conversation_history()

    return f"###Question\n{user_query}\n###Conversation history\n{conversation_history}\n###context\n{context_text}"



In [10]:
# Generating the answers:

@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=1,min=1,max=10),
    before_sleep=before_sleep_log(logger,logging.WARNING)
)
def generate_answer(user_message,system_message):
    response = client.chat.completions.create(
                    model=bundle["model"],
                    messages=[
                        {"role":"system","content":system_message},
                        {"role":"user","content":user_message}
                    ],
                     )

    return response.choices[0].message.content.strip()





In [11]:
# will use this function in case want to see the metadata for inquiry
def rag_answer(user_query, retriever):
    """End-to-end: retrieve → build messages → generate → return audit bundle."""
    retrieved = retrieve_chunks
    user_message = build_user_message(user_query, retrieved)
    answer = generate_answer(SYSTEM_MESSAGE)
    return {"answer": answer, "retrieved_chunks": retrieved, "user_message": user_message}


In [12]:
def save_conversation_history(history):
    """Saves the current conversation history to a JSON file."""
    with open(HISTORY_FILE, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"Conversation history saved to {HISTORY_FILE}.")


In [20]:
def load_eval(questions):
    with open(questions) as f:
        queries=json.load(f)
    return queries
print(load_eval(r"..\testing\eval.json"))
evaluation=load_eval(r"..\testing\eval.json")

[{'id': 1, 'difficulty': 'Easy', 'topic': 'Architecture', 'question': 'What was the purpose of the Great Bath at Mohenjo-daro?', 'expected_answer': 'The Great Bath was a large public bathing structure, likely used for ritual or ceremonial bathing.'}, {'id': 2, 'difficulty': 'Easy', 'topic': 'Cities', 'question': 'Name two major cities of the Indus Civilization discussed in the book.', 'expected_answer': 'Harappa and Mohenjo-daro.'}, {'id': 3, 'difficulty': 'Medium', 'topic': 'Trade', 'question': 'How did the Indus Civilization conduct trade with other regions?', 'expected_answer': 'The civilization traded using land and sea routes with regions such as Mesopotamia, transporting goods like beads, metals and textiles.'}, {'id': 4, 'difficulty': 'Medium', 'topic': 'Urban Planning', 'question': 'What features of Indus cities demonstrate advanced urban planning?', 'expected_answer': 'Grid-pattern streets, drainage systems, standardized bricks, wells, and organized residential layouts demonst

In [24]:
# --- Multi-turn RAG interaction loop ---

def main():

    conversation_history = load_conversation_history()
    turn_count = 0
    MAX_STEPS = 5
    STOP_WORDS = ["exit", "quit", "stop"]

    print("\n--- Starting Multi-Turn RAG Chat ---")
    print("Type 'exit', 'quit', or 'stop' to end the conversation.")
    print(f"Maximum {MAX_STEPS} turns allowed per session (excluding system message and previous turns).")



    while turn_count < MAX_STEPS:
        for test in evaluation:

            user_input = test["question"]

            if user_input.lower() in STOP_WORDS:
                print("Exiting chat. Goodbye!")
                break

            # 1. Retrieve relevant documents from Vector DB
            print("--> Retrieving documents for context...")
            context =retrieve_chunks(user_input,retriever)
            user_message_content = build_user_message(user_input, context)
            SYSTEM_MESSAGE = loading_prompt(bundle).strip()

            # 2. Append the user's message (with context) to history
            conversation_history.append({'role': 'user', 'content': user_message_content})

            # 3. Call the LLM with the entire history
            try:
                print("--> Calling LLM...")

                assistant_response = generate_answer(user_message_content,SYSTEM_MESSAGE)

            except Exception as e:
                assistant_response = f'Sorry, I encountered the following error: \n {e}'
                print(f"LLM Error: {e}")

            # 4. Append the LLM's response to history
            conversation_history.append({'role': 'assistant', 'content': assistant_response})

            # 5. Save the updated history
            save_conversation_history(conversation_history)

            print(f"Assistant: {assistant_response}")
            test["Assistant_Response"]=assistant_response
            for chunk in context:
                print(f"Source : {chunk["Metadata"].get("page")}")
                test["Source"] =chunk["Metadata"].get("page")
                break
            with open(r"..\testing\result.json","w") as f:
                json.dump(evaluation,f, indent=4, ensure_ascii=False)
            turn_count += 1

    if turn_count == MAX_STEPS:
        print(f"\nMaximum turns ({MAX_STEPS}) reached. Chat session ended.")

main()






Loaded conversation history from d:\Sindhu-RAG\Storage\conversation\conversation_history.json. Current turns: 10

--- Starting Multi-Turn RAG Chat ---
Type 'exit', 'quit', or 'stop' to end the conversation.
Maximum 5 turns allowed per session (excluding system message and previous turns).
--> Retrieving documents for context...
Loaded conversation history from d:\Sindhu-RAG\Storage\conversation\conversation_history.json. Current turns: 10
--> Calling LLM...
Conversation history saved to d:\Sindhu-RAG\Storage\conversation\conversation_history.json.
Assistant: The Great Bath at Mohenjo-daro was likely related to the religious life of the city or its rulers, and its elaborate and prominent position on the citadel suggests it had an official status, possibly for ceremonial cleansings.
Source : 59
--> Retrieving documents for context...
Loaded conversation history from d:\Sindhu-RAG\Storage\conversation\conversation_history.json. Current turns: 2
--> Calling LLM...
Conversation history save